In [9]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
from scipy.special import gamma, gammaincc
import numpy as np
import math


# -------------- data --------------
new_df = pd.read_csv(
    '/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv',
    sep=','
)

N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 7
Y = 365
Delta_P = [
    round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3)
    for i in range(N)
]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158
gamma_beta  = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000

mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    upper_gamma = gammaincc(gamma_alpha, gamma_beta * age) * gamma(gamma_alpha)
    h_i_age = (
        gamma_beta**gamma_alpha
        * age**(gamma_alpha - 1)
        * np.exp(-gamma_beta * age)
    ) / upper_gamma
    return h_i_age


# =========================================================
# 用固定 C_leak 建模
# 和你给的 cost analysis 设置方式一致
# =========================================================
def solve_model_with_C_leak(C_leak):
    mdl = Model("leak_decision_final")
    mdl.Params.OutputFlag = 1
    mdl.Params.MIPGap = 0.05
    mdl.Params.TimeLimit = 300
    mdl.Params.IntFeasTol = 1e-6
    mdl.Params.NumericFocus = 2

    # -------------- decision variables --------------
    x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
    x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
    alpha   = mdl.addVars(N, T_years + 1, lb=0, name="alpha")
    last_action = mdl.addVars(N, T_years + 1, lb=0, name="last_action")
    cum_leak    = mdl.addVars(N, T_years, lb=0, name="cum_leak")
    yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

    # initialization
    for i in range(N):
        mdl.addConstr(
            alpha[i, 0] ==
            alpha_init[i] / compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta)
            * compute_gamma_h(Age_2023[i] + 1, gamma_alpha, gamma_beta)
        )
        mdl.addConstr(last_action[i, 0] == 0)

    # alpha evolution
    for i in range(N):
        for t in range(1, T_years + 1):
            growth = (
                alpha[i, 0] / compute_gamma_h(Age_2023[i]-1, gamma_alpha, gamma_beta)
            ) * compute_gamma_h(Age_2023[i] + t - 1, gamma_alpha, gamma_beta)

            no_action = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])

            mdl.addConstr(alpha[i, t] >= growth - M_alpha * (1 - no_action))
            mdl.addConstr(alpha[i, t] <= growth + M_alpha * (1 - no_action))

            coper_target = alpha[i, t - 1] / mu_COPER
            mdl.addConstr(alpha[i, t] <= coper_target + M_alpha * (1 - x_COPER[i, t - 1]))
            mdl.addConstr(alpha[i, t] >= coper_target - M_alpha * (1 - x_COPER[i, t - 1]))

            mant_target = alpha[i, t - 1] / mu_MANT
            mdl.addConstr(alpha[i, t] <= mant_target + M_alpha * (1 - x_MANT[i, t - 1]))
            mdl.addConstr(alpha[i, t] >= mant_target - M_alpha * (1 - x_MANT[i, t - 1]))

    # last_action update
    M_time = T_years + 5
    for i in range(N):
        for t in range(1, T_years + 1):
            action_t1 = x_COPER[i, t - 1] + x_MANT[i, t - 1]

            mdl.addConstr(last_action[i, t] >= (t - 1) - M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] <= (t - 1) + M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] >= last_action[i, t - 1] - M_time * action_t1)
            mdl.addConstr(last_action[i, t] <= last_action[i, t - 1] + M_time * action_t1)

    # yearly leak calculation
    BIG_M_LEAK = 1e6
    for i in range(N):
        for t in range(T_years):
            mdl.addConstr(yearly_leak[i, t] == (Y * alpha[i, t] * (10 ** 5)) * K[i])

    # cumulative leak logic
    LEAK_TH = 2.0
    for i in range(N):
        mdl.addConstr(cum_leak[i, 0] == yearly_leak[i, 0])

        for t in range(1, T_years):
            no_action_prev = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])
            action_prev    = x_COPER[i, t - 1] + x_MANT[i, t - 1]

            mdl.addConstr(
                cum_leak[i, t] >= cum_leak[i, t - 1] + yearly_leak[i, t]
                - BIG_M_LEAK * (1 - no_action_prev)
            )
            mdl.addConstr(
                cum_leak[i, t] <= cum_leak[i, t - 1] + yearly_leak[i, t]
                + BIG_M_LEAK * (1 - no_action_prev)
            )

            mdl.addConstr(
                cum_leak[i, t] >= yearly_leak[i, t]
                - BIG_M_LEAK * (1 - action_prev)
            )
            mdl.addConstr(
                cum_leak[i, t] <= yearly_leak[i, t]
                + BIG_M_LEAK * (1 - action_prev)
            )

        for t in range(T_years):
            x_this_year = x_COPER[i, t] + x_MANT[i, t]
            mdl.addConstr(x_this_year <= 1)
            mdl.addConstr(cum_leak[i, t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

    # objective: 使用固定 C_leak
    leak_cost = quicksum(
        yearly_leak[i, t] * C_leak
        for i in range(N) for t in range(T_years)
    )

    action_cost = quicksum(
        C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t]
        for i in range(N) for t in range(T_years)
    )

    mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

    mdl.optimize()

    if mdl.status == GRB.OPTIMAL or mdl.status == GRB.TIME_LIMIT:
        print(f"\n✅ C_leak = {C_leak:,.0f} | Total Cost: {mdl.objVal:,.0f}")

        print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
        total_leak_cost = 0
        total_action_cost = 0

        for i in range(N):
            act_type = "No action"
            print(f"\n{'='*70}")
            print(
                f"CSEM {i+1} "
                f"(Initial α={alpha_init[i]:.6f}, Volume={V[i]:.1f}m³, "
                f"csemCur={new_df['csemCur'].iloc[i]:.1f}, Age2023={Age_2023[i]:.1f} years), "
                f"K={K[i]:.6f}, ΔP={Delta_P[i]:.3f}"
            )
            print(f"{'='*70}")

            csem_leak_cost = 0
            csem_action_cost = 0

            for t in range(T_years):
                if t == 0:
                    alpha_start = alpha_init[i]
                    alpha_end   = alpha[i, 0].X
                    leak_kg     = yearly_leak[i, t].X
                    leak_cost_t = leak_kg * C_leak
                    csem_leak_cost += leak_cost_t
                else:
                    alpha_start = alpha[i, t - 1].X
                    alpha_end   = alpha[i, t].X
                    leak_kg     = yearly_leak[i, t].X
                    leak_cost_t = leak_kg * C_leak
                    csem_leak_cost += leak_cost_t

                    act_type = ""
                    act_cost = 0
                    if x_COPER[i, t - 1].X > 0.5:
                        act_type = "COPER"
                        act_cost = C_COPER
                        csem_action_cost += act_cost
                    elif x_MANT[i, t - 1].X > 0.5:
                        act_type = "MANT"
                        act_cost = C_MANT
                        csem_action_cost += act_cost
                    else:
                        act_type = "No action"

                    total_leak_cost += leak_cost_t
                    if act_cost > 0:
                        total_action_cost += act_cost

                alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
                cum_leak_val = cum_leak[i, t].X
                valid_check = "❌" if (cum_leak_val >= 2) else "✅"
                cr_display = 365 * alpha_end / Delta_P[i] if Delta_P[i] > 1e-9 else 0

                print(
                    f"  Y{t+1:2d}: {act_type:<8} | "
                    f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                    f"(-{alpha_reduction:5.1f}%) | "
                    f"Leak{leak_kg:>6.3f}kg | "
                    f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check} | "
                    f"CR {cr_display:.3f}"
                )

            print(
                f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + "
                f"Leak cost{csem_leak_cost:>10,.0f} = "
                f"Total {csem_action_cost + csem_leak_cost:>11,.0f}"
            )
            print(f"  Total leak: {sum(yearly_leak[i, t].X for t in range(T_years)):.1f} kg")
            print(f"  Total COPER action: {sum(x_COPER[i, t].X for t in range(T_years)):.0f}")
            print(f"  Total MANT action: {sum(x_MANT[i, t].X for t in range(T_years)):.0f}")

        total_coper_actions = sum(x_COPER[i, t].X for i in range(N) for t in range(T_years))
        total_mant_actions  = sum(x_MANT[i, t].X for i in range(N) for t in range(T_years))
        total_leak_kg       = sum(yearly_leak[i, t].X for i in range(N) for t in range(T_years))
        model_leak_cost     = mdl.objVal - total_action_cost

        print(f"\n{'='*70}")
        print(
            f"🎯 Total: action cost {total_action_cost:>12,.0f} + "
            f"leak cost {model_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}"
        )
        print(f"{'='*70}")
        print(f"Total leak over entire horizon: {total_leak_kg:.1f} kg")
        print(f"Total COPER actions: {total_coper_actions:.0f}")
        print(f"Total MANT actions: {total_mant_actions:.0f}")

        return {
            "C_leak": C_leak,
            "total_cost": mdl.objVal,
            "total_leak_kg": total_leak_kg,
            "leak_cost": model_leak_cost,
            "action_cost": total_action_cost,
            "total_coper_actions": total_coper_actions,
            "total_mant_actions": total_mant_actions,
            "model_leak_cost": model_leak_cost,
            "status": mdl.status
        }

    else:
        print(f"❌ Optimization status with C_leak={C_leak}: {mdl.status}")
        return {
            "C_leak": C_leak,
            "total_cost": None,
            "leak_cost": None,
            "action_cost": None,
            "total_coper_actions": None,
            "total_mant_actions": None
        }


# =========================================================
# 参数扫描（完全按你给的设置方式）
# =========================================================
base_C_leak = 25200 * 0.100
C_leak_list = [
    base_C_leak * 0.01,
    base_C_leak * 0.1,
    base_C_leak * 1,
    base_C_leak * 2.0,
    base_C_leak * 3.0,
    base_C_leak * 4.0,
    base_C_leak * 5.0,
    base_C_leak * 10.0,
    base_C_leak * 30.0,
    base_C_leak * 50.0,
    base_C_leak * 70.0,
    base_C_leak * 100.0,
    base_C_leak * 1000.0,
    base_C_leak * 10000.0,
    base_C_leak * 100000.0
]

results = []
for C_leak in C_leak_list:
    res = solve_model_with_C_leak(C_leak)
    results.append(res)

pd.set_option("display.float_format", "{:,.0f}".format)
result_df = pd.DataFrame(results)

print("\n========== Summary of Sensitivity Analysis ==========")
print(result_df)

result_df.to_csv("sensitivity_analysis_C_leak_alpha.csv", index=False)

print('number of coper actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_coper_actions']:.0f} COPER actions")

print('number of mant actions for each C_LEAK:')
for res in results:
    print(f"  C_leak={res['C_leak']:,.3f}: {res['total_mant_actions']:.0f} MANT actions")

Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 26432 rows, 9856 columns and 89376 nonzeros
Model fingerprint: 0x8a3a88f4
Variable types: 6720 continuous, 3136 integer (3136 binary)
Coefficient statistics:
  Matrix range     [3e-02, 1e+06]
  Objective range  [3e+01, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-07, 1e+06]
Presolve removed 26341 rows and 9824 columns
Presolve time: 0.18s
Presolved: 91 rows, 32 columns, 398 nonzeros
Variable types: 18 continuous, 14 integer (14 binary)
Found heuristic solution: objective 67595.198328

Explored 0 nodes (0 simplex iterations) in 0.18 seconds (0.35 work units)
Thread count was 12 (of 12 available processors)

Solution count 1: 67595.2 



In [7]:


from gurobipy import Model, GRB, quicksum
import pandas as pd
from scipy.special import gamma, gammaincc
import numpy as np
import math

# -------------- data --------------
new_df = pd.read_csv(
    '/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv',
    sep=','
)

N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 7
Y = 365
Delta_P = [
    round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3)
    for i in range(N)
]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158
gamma_beta  = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000

# 原始 yearly leak unit cost
base_c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
base_C_leak_t = [c * 25200 for c in base_c_factors]

mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    upper_gamma = gammaincc(gamma_alpha, gamma_beta * age) * gamma(gamma_alpha)
    h_i_age = (
        gamma_beta**gamma_alpha
        * age**(gamma_alpha - 1)
        * np.exp(-gamma_beta * age)
    ) / upper_gamma
    return h_i_age

v = compute_gamma_h(Age_2023[0]-1, gamma_alpha, gamma_beta)
print(v)

0.1455258822272107
